In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


: 

In [ ]:
# read csv file with pandas
df = pd.read_csv('insurance.csv')
df

# EDA

In [ ]:
# return all info about data
df.info()

In [ ]:
# read first 5 rows with head()
df.head()

In [ ]:
# read last 5 rows with tail()
df.tail()

In [ ]:
# description of data (will work for numeric values only)
df.describe()

In [ ]:
df.dtypes

In [ ]:
# return shape of data
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
# return all columns names
df.columns

In [ ]:
numeric_columns = ['age', 'bmi', 'children','charges']
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col],kde=True,bins=20)

In [ ]:
sns.countplot(x = df['children'])

In [ ]:
sns.countplot(x = df['sex'])

In [ ]:
sns.countplot(x = df['smoker'])

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x = df[col])

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True)

# Data Cleaning

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.head()

In [ ]:
# Remove duplicate data
print(df_cleaned.shape)
df_cleaned.drop_duplicates(inplace=True)
print(df_cleaned.shape)

In [ ]:
# Check for null values
df_cleaned.isnull().sum()

In [ ]:
# Check data types for all columns
df_cleaned.dtypes

In [ ]:
# check value count for sex column and then encode it with numeric values
df_cleaned['sex'].value_counts()

In [ ]:
# set 0 for male and 1 for female using label encoding
df_cleaned['sex'] = df_cleaned['sex'].map({"male":0, "female":1})
df_cleaned.head()

In [ ]:
# check value count for smoker column and then encode it with numeric values
df_cleaned['smoker'].value_counts() 

In [ ]:
# set 0 for no and 1 for yes using label encoding
df_cleaned['smoker'] = df_cleaned['smoker'].map({"no":0, "yes":1})
df_cleaned.head()

In [ ]:
# change columns name sex -> is_female and smoker -> is_smoker as per the data
df_cleaned.rename(columns = {
    "sex":"is_female",
    "smoker":"is_smoker"
}, inplace=True)
df_cleaned.head()

In [ ]:
# check value count for region column and then encode it with numeric values
df_cleaned['region'].value_counts()

In [ ]:
# encode region column with one-hot encoding
df_cleaned = pd.get_dummies(df_cleaned,columns=['region'],drop_first=True)
df_cleaned.head()

In [ ]:
# convert this true false values into 0 and 1
df_cleaned = df_cleaned.astype('int')
df_cleaned.head()

# Feature Engineering and Extraction

In [ ]:
sns.histplot(df['bmi'])

In [ ]:
# create bmi category column
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins = [0,18.5,24.9,29.9,float('inf')],
    labels = ['Underweight','Normal','Overweight','Obese']
)

In [ ]:
df_cleaned.head()

In [ ]:
# use one-hot encoding for bmi_category
df_cleaned = pd.get_dummies(df_cleaned,columns=['bmi_category'])
df_cleaned = df_cleaned.astype(int)
df_cleaned

In [ ]:
# apply scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
col = ['age','bmi','children']

df_cleaned[col] = scaler.fit_transform(df_cleaned[col])

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.columns

In [ ]:
from scipy.stats import pearsonr

selected_features = [
    'age', 'is_female', 'bmi', 'children', 'is_smoker',
       'region_northwest', 'region_southeast', 'region_southwest',
       'bmi_category_Underweight', 'bmi_category_Normal',
       'bmi_category_Overweight', 'bmi_category_Obese'
]

correlations = {
    feature: pearsonr(df_cleaned[feature],df_cleaned['charges'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()),columns=['feature','Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation',ascending=False)

In [ ]:
from scipy.stats import chi2_contingency